In [1]:
import zipfile
import xml.etree.ElementTree as ET
import csv
from pathlib import Path
import shutil



In [ ]:

# -------------------------------------------
# Extract original images from word/media
# -------------------------------------------
def extract_images(docx_path, out_dir):
    docx_path = Path(docx_path)
    out_dir = Path(out_dir)
    media_dir = out_dir / "media"
    media_dir.mkdir(parents=True, exist_ok=True)

    extracted = []

    with zipfile.ZipFile(docx_path, 'r') as z:
        for file in z.namelist():
            
            if file.startswith("word/media/"):
                filename = Path(file).name
                target = media_dir / filename
                print(f"Target path: {target}")
                with z.open(file) as src, open(target, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                    print(f"Extracted: {file}")

                extracted.append(filename)

    return extracted, media_dir


# -------------------------------------------
# Extract alt text and image relationships
# -------------------------------------------
def extract_alt_text(docx_path):
    docx_path = Path(docx_path)
    alt_entries = []

    with zipfile.ZipFile(docx_path, 'r') as z:

        # read the main document XML
        document_xml = z.read("word/document.xml")
        root = ET.fromstring(document_xml)

        # Word XML namespaces
        ns = {
            "w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main",
            "wp": "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing",
            "a":  "http://schemas.openxmlformats.org/drawingml/2006/main",
            "pic": "http://schemas.openxmlformats.org/drawingml/2006/picture"
        }

        # Read relationships for images (rels file)
        rels = {}
        rels_xml = z.read("word/_rels/document.xml.rels")
        rels_root = ET.fromstring(rels_xml)

        for rel in rels_root.findall("Relationship", {"": "http://schemas.openxmlformats.org/package/2006/relationships"}):
            if rel.attrib.get("Type", "").endswith("/image"):
                rels[rel.attrib["Id"]] = Path(rel.attrib["Target"]).name

        # Scan for pictures in document.xml
        for pic in root.findall(".//pic:pic", ns):
            # alt text is stored in <pic:cNvPr>
            cnvpr = pic.find("pic:nvPicPr/pic:cNvPr", ns)
            alt_text = cnvpr.attrib.get("descr", "")
            name = cnvpr.attrib.get("name", "")

            # find relationship id
            blip = pic.find(".//a:blip", ns)
            if blip is not None:
                embed = blip.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}embed")
                image_file = rels.get(embed, "UNKNOWN")
            else:
                image_file = "UNKNOWN"

            alt_entries.append({
                "image_file": image_file,
                "image_name": name,
                "alt_text": alt_text
            })

    return alt_entries


# -------------------------------------------
# Main pipeline
# -------------------------------------------
def run_pipeline(docx_path, out_folder, csv_out):
    docx_path = Path(docx_path)
    out_folder = Path(out_folder)
    out_folder.mkdir(exist_ok=True)

    # Extract images
    images, media_path = extract_images(docx_path, out_folder)

    # Extract alt text data
    alt_data = extract_alt_text(docx_path)

    # Write CSV
    headers = ["image_file", "image_name", "alt_text", "file_saved_path"]

    with open(csv_out, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()

        for entry in alt_data:
            filename = entry["image_file"]
            file_path = media_path / filename
            writer.writerow({
                "image_file": filename,
                "image_name": entry["image_name"],
                "alt_text": entry["alt_text"],
                "file_saved_path": str(file_path)
            })

    print("\n--- DONE ---")
    print("Images extracted to:", media_path)
    print("CSV written to:", csv_out)



# -------------------------------------------
# USER SETTINGS
# -------------------------------------------
if __name__ == "__main__":
    # Change these paths:
    DOCX_FILE = r"/home/joe/work/NotBic/Ports/BTS_Port-Performance-2026_Annual-Report_Draft for Review_12.1.25.docx"
    OUTPUT_FOLDER = r"word_images"
    CSV_OUTPUT = r"word_images/docx_image_map.csv"

    run_pipeline(DOCX_FILE, OUTPUT_FOLDER, CSV_OUTPUT)


Extracted: word/media/image1.png
Extracted: word/media/image2.png
Extracted: word/media/image3.png
Extracted: word/media/image4.png
Extracted: word/media/image5.png
Extracted: word/media/image6.png
Extracted: word/media/image7.png
Extracted: word/media/image8.png
Extracted: word/media/image9.png
Extracted: word/media/image10.png
Extracted: word/media/image11.jpeg
Extracted: word/media/image12.png
Extracted: word/media/image13.png
Extracted: word/media/image14.png
Extracted: word/media/image15.png
Extracted: word/media/image16.png
Extracted: word/media/image17.png
Extracted: word/media/image18.png

--- DONE ---
Images extracted to: word_images/media
CSV written to: word_images/docx_image_map.csv
